# Checkpoint 5 — Explore JSON as tables

A pandas **DataFrame** is an in-memory table: named columns and rows you can inspect, filter, and summarize. JSON is the storage format; tables provide an analysis view. No source files are replaced or rewritten.

Run independently using `.venv`. The notebook extra now includes pandas. All original records, duplicates, revisions, and file order are preserved. Each table gets a one-based `file_row` for traceability; this is metadata, not a model feature.

Keep telemetry, decision checkpoints, and audited incidents separate: blindly joining incidents onto telemetry could expose future information or multiply rows. These are exploration tables, not a model-ready training dataset.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from inside the candidate repository.")

def load_records(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

raw_events = load_records("events.jsonl")
raw_decisions = load_records("decision_times.jsonl")
raw_labels = load_records("labels.jsonl")
manifest = json.loads((ROOT / "data/MANIFEST.json").read_text())

def as_table(records, timestamp_columns):
    # Preserve original fields; flatten nested objects into additional dotted columns.
    table = pd.DataFrame(records)
    flattened = pd.json_normalize(records, sep=".")
    for column in flattened.columns:
        if column not in table.columns:
            table[column] = flattened[column]
    table.insert(0, "file_row", range(1, len(table) + 1))
    for column in timestamp_columns:
        if column in table.columns:
            # Retain source text and add timezone-aware UTC analysis columns.
            table[column + "_utc"] = pd.to_datetime(table[column], utc=True, format="ISO8601", errors="raise")
    return table

events_df = as_table(raw_events, ["device_time", "received_at"])
decisions_df = as_table(raw_decisions, ["decision_time"])
labels_df = as_table(raw_labels, ["incident_at", "label_available_at"])
manifest_df = pd.DataFrame([manifest])
print("Tables loaded without sorting, deduplicating, joining, or modifying source files.")


Tables loaded without sorting, deduplicating, joining, or modifying source files.


## 1. Table inventory and previews

The previews show only the first few rows; the DataFrame variables contain all rows. Event payload objects remain intact, with extra columns such as `payload.firmware` for convenient filtering. Original timestamp text remains alongside parsed UTC columns.


In [2]:
tables = {"events": events_df, "decision_times": decisions_df,
          "labels": labels_df, "manifest": manifest_df}
display(pd.DataFrame([{"table": name, "rows": len(table), "columns": len(table.columns)}
                      for name, table in tables.items()]))
for name, table in tables.items():
    print("\n" + name.upper())
    display(table.head(5))


            table   rows  columns
0          events  11019       14
1  decision_times   1800        4
2          labels    133        8
3        manifest      1        5

EVENTS
   file_row  ...           received_at_utc
0         1  ... 2026-01-01 03:35:00+00:00
1         2  ... 2026-01-01 01:01:00+00:00
2         3  ... 2026-01-01 02:02:00+00:00
3         4  ... 2026-01-01 00:05:00+00:00
4         5  ... 2026-01-01 03:35:00+00:00

[5 rows x 14 columns]

DECISION_TIMES
   file_row         decision_time shipment_id         decision_time_utc
0         1  2026-01-01T08:00:00Z     s-00000 2026-01-01 08:00:00+00:00
1         2  2026-01-01T11:00:00Z     s-00000 2026-01-01 11:00:00+00:00
2         3  2026-01-01T14:00:00Z     s-00000 2026-01-01 14:00:00+00:00
3         4  2026-01-01T11:00:00Z     s-00001 2026-01-01 11:00:00+00:00
4         5  2026-01-01T14:00:00Z     s-00001 2026-01-01 14:00:00+00:00

LABELS
   file_row  ...    label_available_at_utc
0         1  ... 2026-01-02 11:00:00+00:00

## 2. Inspect column types and missing fields

A missing payload field is not the same thing as a missing temperature. Blank entries may mean a field is not applicable. We inspect missingness before deciding how to represent it to a model.


In [3]:
for name, table in tables.items():
    print("\n" + name.upper() + " — column audit")
    display(pd.DataFrame({"dtype": table.dtypes.astype(str), "missing_rows": table.isna().sum()}))



EVENTS — column audit
                                  dtype  missing_rows
file_row                          int64             0
device_time                         str             0
event_id                            str             0
kind                                str             0
payload                          object             0
received_at                         str             0
revision                          int64             0
shipment_id                         str             0
source                              str             0
value                           float64             0
payload.firmware                    str           133
payload.correction                  str         10933
device_time_utc     datetime64[us, UTC]             0
received_at_utc     datetime64[us, UTC]             0

DECISION_TIMES — column audit
                                 dtype  missing_rows
file_row                         int64             0
decision_time                 

## 3. Ask simple analysis questions

Which event types are present? How often do identities repeat? How delayed are deliveries? Here we count repeated identities without removing them. The delay column includes correction availability lag and relies on the device clock; it is not guaranteed pure network latency.

Counts describe the supplied synthetic dataset. They are not fixed assumptions for the implementation.


In [4]:
display(events_df.groupby("kind", dropna=False).size().rename("deliveries").reset_index())
repeated = events_df.duplicated(["event_id", "revision"], keep=False)
print("Rows belonging to repeated identities:", int(repeated.sum()))
print("Extra deliveries after the first identity occurrence:",
      int(events_df.duplicated(["event_id", "revision"]).sum()))
display(events_df.loc[repeated, ["file_row", "event_id", "revision", "value", "received_at"]].head(8))
revision_counts = events_df.groupby("event_id")["revision"].nunique()
print("Event IDs with multiple revisions:", int((revision_counts > 1).sum()))
delay_minutes = (events_df["received_at_utc"] - events_df["device_time_utc"]).dt.total_seconds() / 60
display(delay_minutes.describe().to_frame("revision_arrival_delay_minutes"))


            kind  deliveries
0      door_open         133
1  temperature_c       10886
Rows belonging to repeated identities: 2400
Extra deliveries after the first identity occurrence: 1200
    file_row         event_id  revision  value           received_at
5          6  s-00000-temp-04         1  3.399  2026-01-01T04:01:00Z
6          7  s-00000-temp-04         1  3.399  2026-01-01T04:01:00Z
12        13  s-00001-temp-04         1  3.465  2026-01-01T07:05:00Z
13        14  s-00001-temp-04         1  3.465  2026-01-01T07:05:00Z
25        26  s-00002-temp-04         1  4.821  2026-01-01T10:05:00Z
26        27  s-00002-temp-04         1  4.821  2026-01-01T10:05:00Z
31        32  s-00000-temp-11         1  5.926  2026-01-01T11:35:00Z
32        33  s-00000-temp-11         1  5.926  2026-01-01T11:35:00Z
Event IDs with multiple revisions: 86
       revision_arrival_delay_minutes
count                    11019.000000
mean                        46.440149
std                         93.144317

## 4. Filter one shipment without losing delivery order

The table below retains file order. The checkpoint table tells us when predictions were requested; the incident table provides later audited outcomes. Viewing them together does not authorize using outcomes as features.


In [5]:
chosen = decisions_df.sort_values(["decision_time_utc", "shipment_id"]).iloc[0]["shipment_id"]
print("Selected shipment:", chosen)
display(events_df.loc[events_df["shipment_id"] == chosen,
                      ["file_row", "event_id", "revision", "device_time_utc", "received_at_utc", "kind", "value"]])
display(decisions_df.loc[decisions_df["shipment_id"] == chosen])
display(labels_df.loc[labels_df["shipment_id"] == chosen])

assert len(events_df) == len(raw_events)
assert events_df["event_id"].tolist() == [row["event_id"] for row in raw_events]
assert events_df["revision"].tolist() == [row["revision"] for row in raw_events]
assert events_df["payload"].tolist() == [row["payload"] for row in raw_events]
assert len(decisions_df) == len(raw_decisions) and len(labels_df) == len(raw_labels)
assert events_df["file_row"].tolist() == list(range(1, len(raw_events) + 1))
assert str(events_df["received_at_utc"].dt.tz) == "UTC"
print("Verified: row counts, event order, revisions, original payloads, and UTC timestamps.")


Selected shipment: s-00000
    file_row         event_id  ...           kind  value
0          1  s-00000-temp-03  ...  temperature_c  3.737
1          2  s-00000-temp-01  ...  temperature_c  3.505
2          3  s-00000-temp-02  ...  temperature_c  3.786
3          4  s-00000-temp-00  ...  temperature_c  4.224
5          6  s-00000-temp-04  ...  temperature_c  3.399
6          7  s-00000-temp-04  ...  temperature_c  3.399
8          9  s-00000-temp-05  ...  temperature_c  5.038
10        11  s-00000-temp-06  ...  temperature_c  4.687
15        16  s-00000-temp-07  ...  temperature_c  3.798
23        24  s-00000-temp-10  ...  temperature_c  5.386
27        28  s-00000-temp-08  ...  temperature_c  3.908
31        32  s-00000-temp-11  ...  temperature_c  5.926
32        33  s-00000-temp-11  ...  temperature_c  5.926
33        34  s-00000-temp-09  ...  temperature_c  4.667
37        38  s-00000-temp-12  ...  temperature_c  7.632
39        40  s-00000-temp-13  ...  temperature_c  9.444
45  

## 5. Your next exploration

In a new code cell, `events_df.head(10)` shows the first ten deliveries. To inspect one column, use `events_df["kind"]`. Filtering creates a view for analysis; avoid overwriting the original full table or dropping duplicates before understanding them.

**What changed:** storage records became convenient analysis tables. Feature engineering and point-in-time rules are still required before model training.

**Interview notes:** “I used tabular views to inspect raw telemetry, preserved delivery indices and revision identities, and kept labels separate to avoid introducing future outcomes into the features.”

**Next:** review the table columns together, then return to label eligibility and evaluation cutoffs. No model has been trained.


## 6. Save the analysis tables to the data folder

Export four CSV files to `personal/data/tables/`. Running this cell regenerates them from the DataFrames above. CSV preserves rows and columns but not pandas data types; parse the `_utc` columns as UTC when loading them again. Empty CSV fields represent missing values in these analysis exports. The original JSON remains authoritative for exact types and null/empty-string distinctions.

Serialize nested payload objects as valid JSON text rather than Python dictionary text. Keep file-row tracking, duplicates, revisions, and all rows in their current original order.


In [6]:
import csv

export_dir = ROOT / 'personal' / 'data' / 'tables'
export_dir.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    exported = table.copy()
    for column in exported.columns:
        if exported[column].dtype == "object":
            exported[column] = exported[column].map(
                lambda value: json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
                if isinstance(value, (dict, list)) else value)
    path = export_dir / f"{name}.csv"
    exported.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        saved = list(reader)
        assert reader.fieldnames == list(table.columns)
    assert len(saved) == len(table)
    if "file_row" in table.columns:
        assert [int(row["file_row"]) for row in saved] == table["file_row"].tolist()
    if name == "events":
        assert [row["event_id"] for row in saved] == table["event_id"].tolist()
        assert [int(row["revision"]) for row in saved] == table["revision"].tolist()
        assert [json.loads(row["payload"]) for row in saved] == table["payload"].tolist()
    print(f"Saved {path.relative_to(ROOT)}: {len(saved):,} rows")


Saved data/tables/events.csv: 11,019 rows
Saved data/tables/decision_times.csv: 1,800 rows
Saved data/tables/labels.csv: 133 rows
Saved data/tables/manifest.csv: 1 rows
